In [1]:
pip install skfuzzy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement skfuzzy (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for skfuzzy


In [11]:
import subprocess
import sys

# List of required libraries
libraries = [
    "numpy",
    "scikit-learn",
    "tensorflow",
    "tensorflow-directml"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Error installing {lib}: {e}")

print("\nAll required libraries installed successfully!")
print("Your environment is ready for the CNN model.")

Installing required libraries...

numpy installed successfully!
scikit-learn installed successfully!
tensorflow installed successfully!
Error installing tensorflow-directml: Command '['c:\\Users\\chait\\OneDrive\\Desktop\\CNS\\vitamin-deficiency-main\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'tensorflow-directml']' returned non-zero exit status 1.

All required libraries installed successfully!
Your environment is ready for the CNN model.


In [5]:
import numpy as np
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [6]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 32
# Define directories for training and testing data
train_data_dir = "../dataset/train"
test_data_dir = "../dataset/test"

In [7]:
# Data augmentation for training images
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rescale=1./255  # Normalize pixel values
)

# Data augmentation for testing images (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)


In [8]:
# Create data generators for training and testing
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True  # No need to shuffle for evaluation
)



Found 1174 images belonging to 14 classes.
Found 428 images belonging to 14 classes.


# TRAINING PHASE

In [9]:
# Define the VGG16 base model
from tensorflow.keras.applications import EfficientNetV2L

def create_EfficientNetV2L_model():
    EfficientNetV2L_model = EfficientNetV2L(
        weights='imagenet', 
        include_top=False, 
        input_shape=(img_height, img_width, 3)
    )
    EfficientNetV2L_model.trainable = False


    model = Sequential([
        EfficientNetV2L_model,
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(14, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(model.summary())
    return model




In [10]:
#MODEL fit
model = create_EfficientNetV2L_model()
model.fit(train_generator, epochs=20, validation_data=test_generator)


473176280/473176280 ━━━━━━━━━━━━━━━━━━━━ 134s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-l (Functional)   │ (None, 4, 4, 1280)     │   117,746,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,829,998 (449.49 MB)

 Trainable params: 83,022 (324.30 KB)

 Non-trainable params: 117,746,976 (449.17 MB)

None
Epoch 1/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 155s 3s/step - accuracy: 0.0733 - loss: 3.0451 - val_accuracy: 0.0421 - val_loss: 2.6337
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.0843 - loss: 2.8550 - val_accuracy: 0.0397 - val_loss: 2.6835
Epoch 3/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step - accuracy: 0.0860 - loss: 2.8054 - val_accuracy: 0.0491 - val_loss: 2.7163
Epoch 4/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.0997 - loss: 2.7638 - val_accuracy: 0.0491 - val_loss: 2.6599
Epoch 5/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.0860 - loss: 2.7229 - val_accuracy: 0.0748 - val_loss: 2.6028
Epoch 6/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 56s 2s/step - accuracy: 0.1014 - loss: 2.7270 - val_accuracy: 0.0864 - val_loss: 2.5948
Epoch 7/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.1193 - loss: 2.6906 - val_accuracy: 0.0841 - val_loss: 2.5936
Epoch 8/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.1031 - loss: 2.6859 - val_accuracy: 0.2150 - val

In [12]:
# predicting train dataset 
test_pridiction = model.predict(test_generator)

# true labels 
test_true_labels = test_generator.classes




14/14 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step 


In [13]:
print( test_pridiction.shape)
print( test_true_labels.shape)

#calculating accuracy
accuracy = np.mean(np.argmax(test_pridiction, axis=1) == test_true_labels)
print(accuracy)


(428, 14)
(428,)
0.1705607476635514


In [14]:
# Save the trained model
model.save("../model_saved_files/[model_name].h5")
print("Model saved successfully!")

Model saved successfully!
